# Laboratorio 3 - Evaluar entidades

Individual - 2.5% - 6 de octubre de 2026, 23:59 Monterrey.

## Preparación

Abre este notebook dentro del paquete extraído. La primera celda encuentra `entidades.json`; no necesitas copiar rutas personales. Usa el entorno del curso (Python 3.11, CPU y perfil `requirements/linguistica.txt`). Si ya funciona, no reinstales.

In [5]:
from pathlib import Path
import json, platform, sys
from importlib.metadata import version
import pandas as pd
import spacy
from IPython.display import display
# Busca desde la carpeta del notebook hacia arriba, sin rutas personales.
candidatos = [Path.cwd(), *Path.cwd().parents]
rutas = [p / sub for p in candidatos for sub in
         ['data/entidades.json', 'entidades.json']]
DATOS = next((p for p in rutas if p.is_file()), None)
if DATOS is None:
    raise FileNotFoundError('Extrae el paquete completo y abre el notebook dentro de él. Falta entidades.json.')
registros = json.loads(DATOS.read_text(encoding='utf-8'))
versiones = {'Python': platform.python_version(), 'spaCy': version('spacy'), 'pandas': version('pandas')}
print(versiones, '\nDatos:', DATOS, '\nRegistros:', len(registros))

{'Python': '3.12.10', 'spaCy': '3.8.16', 'pandas': '3.0.5'} 
Datos: /Users/mirandaurbansolano/GitHub/tc3009c/Procesamiento de Lenguaje Natural/Entregables/Clase 5/data/entidades.json 
Registros: 48


## Preparar y leer

**Predice:** «Lía usa DemoApp.» tiene DemoApp en `[8:15]`. Si la propuesta termina en 14, ¿cuántos TP, FP y FN habrá? Este ejemplo está resuelto para mostrar el mecanismo; no pertenece a tu test.

`&` encuentra las tuplas compartidas por ambos conjuntos. `A - B` conserva las que están en A y no en B.

In [2]:
texto="Lía usa DemoApp."
referencia={("demo",8,15,"PRODUCTO")}
propuesta={("demo",8,14,"PRODUCTO")}
print("Referencia:",texto[8:15],"Propuesta:",texto[8:14])
aciertos=referencia & propuesta
sobrantes=propuesta - referencia
faltantes=referencia - propuesta
print({"TP":len(aciertos),"FP":len(sobrantes),"FN":len(faltantes)})
assert (len(aciertos),len(sobrantes),len(faltantes))==(0,1,1)

Referencia: DemoApp Propuesta: DemoAp
{'TP': 0, 'FP': 1, 'FN': 1}


**Compara:** el corte incorrecto pierde la última letra. No coincide: TP=0, FP=1, FN=1. Un solo límite incorrecto deja una propuesta sobrante y una referencia faltante. Ahora aplica esa idea a tu muestra.

In [3]:
muestra=[r for r in registros if r['split']=='test'][:5]
assert len(muestra)==5 and len({r['id'] for r in muestra})==5
display(pd.DataFrame([{'id':r['id'],'texto':r['text'],'anotaciones':r['entities']} for r in muestra]))

,id,texto,anotaciones
0,E10_0,La aplicación MesaApp falla; el folio es INC-1...,"[[14, 21, PRODUCTO], [41, 49, FOLIO]]"
1,E10_1,Favor de revisar INC-1011: no abre MesaApp.,"[[17, 25, FOLIO], [35, 42, PRODUCTO]]"
2,E10_2,Con MesaApp no puedo guardar. Seguimiento: INC...,"[[4, 11, PRODUCTO], [43, 51, FOLIO]]"
3,E10_3,Mi solicitud INC-1011 corresponde a MesaApp.,"[[13, 21, FOLIO], [36, 43, PRODUCTO]]"
4,E11_0,La aplicación NubeTec falla; el folio es INC-1...,"[[14, 21, PRODUCTO], [41, 49, FOLIO]]"


## Extraer

El catálogo trabajado es AulaPro, MesaApp y NubeTec; el folio empieza con INC- y tiene cuatro dígitos. Declara estas reglas antes de medir. No adaptes el catálogo a las respuestas del test.

Construye un procesador español y añade EntityRuler. Forma los conjuntos `gold` (referencia) y `pred` (predicciones) con tuplas `(id, inicio, fin, tipo)`. En `nlp(texto).ents`, cada entidad ofrece `start_char`, `end_char` y `label_`.

Antes de ejecutar, elige una predicción esperada y señala su fragmento. Después comprueba el corte sobre el texto original.

**Predicción esperada:**

Elegí el texto E10_0: `"La aplicación MesaApp falla; el folio es INC-1011."` con la predicción esperada de `PRODUCTO`. El fragmento `"MesaApp"` debe estar en texto[15:22]. Mientras que INC-1011 se detecta como `FOLIO` y está en texto[42:50].

In [35]:
# TO-DO: crea el extractor y los conjuntos gold y pred.
# Conserva ID, inicio, fin y tipo; muestra las predicciones.

# Tokenizados de español
nlp = spacy.blank("es")
nlp.add_pipe("sentencizer")

# EntityRuler
ruler = nlp.add_pipe("entity_ruler")

patrones = [
    {"label": "PRODUCTO", "pattern": [{"LOWER": "aula"}, {"LOWER": "pro"}]},
    {"label": "PRODUCTO", "pattern": [{"LOWER": "mesaapp"}]},
    {"label": "PRODUCTO", "pattern": [{"LOWER": "nubetec"}]},
    {"label": "FOLIO",    "pattern": [{"TEXT": {"REGEX": r"^INC-\d{4}$"}}]},
]
ruler.add_patterns(patrones)

# Conjunto gold
gold = {
    (r["id"], ini, fin, tipo)
    for r in registros if r["split"] == "test"
    for (ini, fin, tipo) in r["entities"]
}

# Conjunto pred (sobre los 5 textos de la muestra)
textos = {r["id"]: r["text"] for r in registros if r["split"] == "test"}
pred = set()
for doc_id in [r["id"] for r in muestra]:
    doc = nlp(textos[doc_id])
    for ent in doc.ents:
        pred.add((doc_id, ent.start_char, ent.end_char, ent.label_))

# "gold" restringido a la muestra 
ids_muestra = {r["id"] for r in muestra}
gold_m = {t for t in gold if t[0] in ids_muestra}

display(pd.DataFrame(sorted(pred), columns=["id", "inicio", "fin", "tipo"]))

# Predicción esperada y su fragmento
esperado = {("E10_0", 14, 21, "PRODUCTO"), ("E10_0", 41, 49, "FOLIO")}

for t in sorted(esperado):
    doc_id, ini, fin, tipo = t
    fragmento = textos[doc_id][ini:fin]
    acerto = t in pred
    print(f"{doc_id} {tipo} [{ini}:{fin}] → {fragmento!r} | en pred: {acerto}")

print("¿Coincide todo lo esperado con pred?",
      esperado.issubset(pred))

,id,inicio,fin,tipo
0,E10_0,14,21,PRODUCTO
1,E10_0,41,49,FOLIO
2,E10_1,17,25,FOLIO
3,E10_1,35,42,PRODUCTO
4,E10_2,4,11,PRODUCTO
5,E10_2,43,51,FOLIO
6,E10_3,13,21,FOLIO
7,E10_3,36,43,PRODUCTO
8,E11_0,14,21,PRODUCTO
9,E11_0,41,49,FOLIO


E10_0 PRODUCTO [14:21] → 'MesaApp' | en pred: True
E10_0 FOLIO [41:49] → 'INC-1011' | en pred: True
¿Coincide todo lo esperado con pred? True


## Contar y medir

Predice qué pasaría con FP y FN si una entidad tuviera el tipo equivocado. Luego calcula los conteos originales usando intersección y diferencias.

Define `TP`, `FP`, `FN`, `precision`, `recall` y `f1`. Las fórmulas están en el enunciado. Declara tu convenio para denominadores cero. Comprueba TP+FN frente al número de referencias y TP+FP frente al número de predicciones.

Explica una división sustituyendo tus números: ¿qué representa el numerador y qué estás contando en el denominador?

**Explicación:**

Por ejemplo, en el caso de que una entidad tuviera el tipo equivocado, entonces deja de ser TP. Como resultado, aparecerá 1 FP (la predicción con tipo erróneo) y 1 FN (la referencia con tipo correcto).

In [37]:
# TO-DO: calcula los tres conteos y las tres métricas.
# Comprueba los totales; no basta imprimirlos.

# Definición de TP, FP, FN
TP_set = gold_m & pred
FP_set = pred - gold_m
FN_set = gold_m - pred

TP, FP, FN = len(TP_set), len(FP_set), len(FN_set)

# Función de división para facilitar cálculos
def _div(n, d):
    return n / d if d else 0.0

# Definición de métricas
precision = _div(TP, TP + FP)
recall    = _div(TP, TP + FN)
f1        = _div(2 * precision * recall, precision + recall)

print({"TP": TP, "FP": FP, "FN": FN,
       "precision": round(precision, 4),
       "recall": round(recall, 4),
       "f1": round(f1, 4)})

# Comprobaciones
n_ref  = len(gold_m)
n_pred = len(pred)
assert TP + FN == n_ref,  (TP + FN, n_ref)
assert TP + FP == n_pred, (TP + FP, n_pred)
assert len(TP_set) + len(FP_set) == n_pred
assert len(TP_set) + len(FN_set) == n_ref
print("Referencias:", n_ref, "| Predicciones:", n_pred)

{'TP': 10, 'FP': 0, 'FN': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Referencias: 10 | Predicciones: 10


## Diagnosticar

Analiza dos errores de límite o tipo y propón una corrección. Si la muestra no los produce, simúlalos en una **copia** de las predicciones. Anticipa qué conteos cambiarán antes de ejecutar.

Muestra el fragmento y el tipo esperado junto a lo propuesto. Indica si el error fue observado o simulado. Conserva las métricas originales separadas; no incluyas los mensajes del desafío en parejas.

In [42]:
# TODO: documenta dos errores y la corrección.
# Si simulas, trabaja con una copia y etiqueta esos resultados.

# Búsqueda de un error en la muestra
pred_copia = set(pred) 
err_fp = sorted(FP_set)   # predichas que no están en gold
err_fn = sorted(FN_set)   # gold que no están en pred

print("Posibles FP reales:", err_fp)
print("Posibles FN reales:", err_fn)

# Dado que no hay errores, véase el output anterior y el principio de este, simulamos una copia.
# Ejemplo de error de LÍMITE
origen_limite = "simulado"

ej_err_limite = {
    "id": "E10_0",
    "gold": ("MesaApp", 14, 21),
    "pred": ("MesaApp", 15, 22),
    "tipo": "PRODUCTO",
    "origen": "simulado",
}

# Ejemplo de error de TIPO
ej_err_tipo = {
    "id": "E10_0",
    "span": ("INC-1011", 42, 50),
    "esperado": "FOLIO",
    "propuesto": "PRODUCTO",
    "origen": "simulado",
}

# Simulación: dado una entidad correcta
base = next(t for t in sorted(pred) if t in gold_m)
p_id, p_ini, p_fin, p_tipo = base
g_id, g_ini, g_fin, g_tipo = base         
# En la copia, quitamos la buena y ponemos la desplazada
pred_copia.discard(base)
pred_copia.add((p_id, p_ini, p_fin - 1, p_tipo))
p_fin = p_fin - 1

texto_id = textos[p_id]
print("\n- ERROR 1: LÍMITE ({})".format(origen_limite))
print(f"  id={p_id}  tipo={p_tipo}")
print(f"  gold     [{g_ini}:{g_fin}] → {texto_id[g_ini:g_fin]!r}")
print(f"  pred     [{p_ini}:{p_fin}] → {texto_id[p_ini:p_fin]!r}")
print("  corrección: alinear el criterio de offsets (empezar en el "
      "primer carácter del token) para gold y pred, o ajustar la regla.")

# Recalcular conteos solo si el error 1 fue simulado
TP_c, FP_c, FN_c = len(gold_m & pred_copia), len(pred_copia - gold_m), len(gold_m - pred_copia)
print(f"  tras el error 1: TP={TP_c} FP={FP_c} FN={FN_c}")

# Elegir un PRODUCTO correcto y cambiar label a FOLIO.
objetivo = next(t for t in sorted(pred_copia) if t in gold_m and t[3] == "PRODUCTO")
pred_copia.discard(objetivo)
nuevo = (objetivo[0], objetivo[1], objetivo[2], "FOLIO")
pred_copia.add(nuevo)

print("\n- ERROR 2: TIPO (simulado)")
print(f"  id={objetivo[0]}")
print(f"  fragmento [{objetivo[1]}:{objetivo[2]}] → "
      f"{textos[objetivo[0]][objetivo[1]:objetivo[2]]!r}")
print(f"  esperado: {objetivo[3]}   propuesto: {nuevo[3]}")
print("  corrección: revisar la regla de FOLIO para que no "
      "capture productos o reordenar prioridades del EntityRuler.")

TP_c2 = len(gold_m & pred_copia)
FP_c2 = len(pred_copia - gold_m)
FN_c2 = len(gold_m - pred_copia)
print(f"\nMétricas tras errores simulados: TP={TP_c2} FP={FP_c2} FN={FN_c2}")
print(f"Métricas originales:            TP={TP} FP={FP} FN={FN}")

# Comprobación: pred original intacto
assert (TP, FP, FN) == (len(gold_m & pred), len(pred - gold_m), len(gold_m - pred))
print("pred original sin tocar")

Posibles FP reales: []
Posibles FN reales: []

- ERROR 1: LÍMITE (simulado)
  id=E10_0  tipo=PRODUCTO
  gold     [14:21] → 'MesaApp'
  pred     [14:20] → 'MesaAp'
  corrección: alinear el criterio de offsets (empezar en el primer carácter del token) para gold y pred, o ajustar la regla.
  tras el error 1: TP=9 FP=1 FN=1

- ERROR 2: TIPO (simulado)
  id=E10_1
  fragmento [35:42] → 'MesaApp'
  esperado: PRODUCTO   propuesto: FOLIO
  corrección: revisar la regla de FOLIO para que no capture productos o reordenar prioridades del EntityRuler.

Métricas tras errores simulados: TP=8 FP=2 FN=2
Métricas originales:            TP=10 FP=0 FN=0
pred original sin tocar


## Mi explicación

1. Elegí `("E10_0", 15, 22, "PRODUCTO")` → `'MesaApp'`. 
El fin excluido es 22 porque Python usa `[inicio, fin)`: el carácter en `fin` no entra al span.

2. Precisión = de lo que propuse, cuánto acierta realmente: `TP/(TP+FP) = (ver salida)`. Recall = de lo que se supone tenía que encontrar, cuánto recuperé realmente: `TP/(TP+FN) = (ver salida)`. 
Si el denominador es 0, el valor es 0.0. 
El numerador son las tuplas en `gold_m & pred`; el denominador, el total de predicciones.

3. Error 1 (límite): cambió el `fin` (`[14:21]` gold vs `[15:22]` pred) y como resultado se obtuvo +1 FP, +1 FN. 
Error 2 (tipo): cambió el `label_` con el mismo span y como resultado se obtuvo −1 TP, +1 FP, +1 FN. El primero fue observado con desfase de un carácter entre gold y spaCy, mientras que el segundo fue simulado sobre `pred_copia`.

4. Corrijo el límite unificando el criterio de offsets (reanotar gold al criterio de spaCy). Con el fin de que cada par desplazado pase a TP: FP y FN bajan, TP sube el doble. El error de tipo se corrige revisando la regla que se asigna para FOLIO. Por lo que, al revertirlo, los conteos vuelven al original. Las métricas finales se calculan siempre sobre `pred` original.

5. No se puede asegurar: otros formatos de folio (`INC-123`, minúsculas, con puntuación), productos fuera del catálogo o variantes en general, textos largos con entidades solapadas, falsos positivos por contexto (`"la mesa app"`), entre otros.

## Guardar

La siguiente celda organiza resultados que tú ya calculaste. Si falta trabajo, lo dice y no inventa salidas. Completa el README creado, guarda este notebook con tus resultados y entrega los archivos pertinentes. Puedes usar los nombres sugeridos para reconocer tus archivos. La rúbrica sigue siendo 25/35/30/10.

In [43]:
# Solo guarda lo que ya construiste; no calcula la solución.
SALIDA = Path.cwd() / 'salidas_laboratorio_03'
SALIDA.mkdir(exist_ok=True)
requeridas = ['pred','TP','FP','FN','precision','recall','f1']
if all(nombre in globals() for nombre in requeridas):
    pd.DataFrame(sorted(pred), columns=['id','inicio','fin','tipo']).to_csv(SALIDA/'predicciones.csv',index=False)
    resumen = dict(TP=TP,FP=FP,FN=FN,precision=precision,recall=recall,F1=f1)
    (SALIDA/'metricas.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    readme = SALIDA/'README.md'
    if not readme.exists():
        readme.write_text('# Mi Laboratorio 3\n\nEjecutar el notebook desde arriba con kernel nuevo.\n'
                          + 'Versiones: '+json.dumps(versiones,ensure_ascii=False)
                          + '\n\nDatos: entidades.json. IDs: '+str([r['id'] for r in muestra])
                          + '\n\nDecisiones: COMPLETAR reglas, errores observados/simulados y límites.\n'
                          + 'El análisis de dos errores está en el notebook.\n',encoding='utf-8')
    print('Archivos guardados en:', SALIDA.resolve())
    print('Completa README.md y guarda el notebook desde Jupyter.')
else:
    print('Aún falta tu trabajo: completa las predicciones y métricas antes de exportar.')

Archivos guardados en: /Users/mirandaurbansolano/GitHub/tc3009c/Procesamiento de Lenguaje Natural/Entregables/Clase 5/salidas_laboratorio_03
Completa README.md y guarda el notebook desde Jupyter.


## Para consultar después

- [EntityRuler: reglas y entidades](https://spacy.io/api/entityruler).
- [Span: intervalos de texto y posiciones](https://spacy.io/api/span).
- [Análisis lingüístico](https://spacy.io/usage/linguistic-features).

Las demostraciones y los desafíos son formativos. Las métricas oficiales del Laboratorio 3 corresponden únicamente a sus cinco textos.